In [30]:
import os, time, shutil, glob
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.common.exceptions import TimeoutException, NoAlertPresentException, WebDriverException
from datetime import datetime, timedelta

# ── CONFIG ─────────────────────────────────────────────────────
CHROMEDRIVER   = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\chromedriver-win64\chromedriver.exe"
SOURCE_FOLDER  = r"C:\temp\expedia_downloads"
BASE_CAPTURE   = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE"
DIRS = {
    "current_agent"   : os.path.join(BASE_CAPTURE, "current_agent"),
    "lc_rawdata"      : os.path.join(BASE_CAPTURE, "lc_rawdata_in_console"),
    "current_interval": os.path.join(BASE_CAPTURE, "current_interval"),
    "current_iex"     : os.path.join(BASE_CAPTURE, "current_iex"),
}
for d in DIRS.values(): os.makedirs(d, exist_ok=True)

URL_BREAKDOWN     = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentBreakdownRealtimeDashboard"
URL_REALTIME      = "https://console.vap.expedia.com/analytics-console-user-interface/optics/agentRealtime"
URL_SHAREPOINT    = (
    "https://cnxmail-my.sharepoint.com/shared?listurl=https%3A%2F%2Fcnxmail-my%2E"
    "sharepoint%2Ecom%2Fpersonal%2Fahmed_ahmedkamh_concentrix_com%2FDocuments"
    "&id=%2Fpersonal%2Fahmed_ahmedkamh_concentrix_com%2FDocuments"
)
DST_UCP           = os.path.join(BASE_CAPTURE, "EN- UCP.xlsx")
NICE_URL_GENERATE = "https://cnxnice02b.nicecloudsvc.com/wfm/supervisor/reports-generate"
NICE_URL_VIEW     = "https://cnxnice02b.nicecloudsvc.com/wfm/supervisor/reports-view"
NICE_REPORT_URL   = "https://cnxnice02b.nicecloudsvc.com/supv/reportAction.mvc?schRptOid=8aa89bca8b4b614b018d4b88862d476c"
LOGIN_VERIFY_CSS  = "button.settingsButton"
LOGIN_TIMEOUT     = 20

# ── NICE date range (Monday → Sunday this week) ───────────────
_today    = datetime.now()
_monday   = _today - timedelta(days=_today.weekday())
_sunday   = _monday + timedelta(days=6)
NICE_FROM = _monday.strftime("%#m/%#d/%y")
NICE_TO   = _sunday.strftime("%#m/%#d/%y")
NICE_FILE = _monday.strftime("%Y_%m_%d") + ".xlsx"

# ── HELPERS ────────────────────────────────────────────────────
def move_files(keyword, dest_dir):
    moved = 0
    for pat in [f"{SOURCE_FOLDER}\\{keyword}*.csv", f"{SOURCE_FOLDER}\\{keyword}*.xlsx"]:
        for fp in glob.glob(pat):
            if fp.endswith(".crdownload"): continue
            dst = os.path.join(dest_dir, os.path.basename(fp))
            if os.path.exists(dst): os.remove(dst)
            shutil.move(fp, dst)
            print(f"  📁 Moved: {os.path.basename(fp)}"); moved += 1
    if not moved: print(f"  ⚠️ No file '{keyword}*' found")
    return moved

def click_download_csv(driver, wait, keyword=None, timeout=30):
    wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "div.uitk-menu-container[aria-hidden='false']")))
    wait.until(EC.element_to_be_clickable((By.XPATH,
        "//div[contains(@class,'uitk-menu-open')][@aria-hidden='false']"
        "//span[text()='Download CSV']/ancestor::button"))).click()
    print("  ✅ Clicked Download CSV")
    if keyword:
        start = time.time()
        while time.time() - start < timeout:
            matches = [f for f in
                glob.glob(f"{SOURCE_FOLDER}\\{keyword}*.csv") +
                glob.glob(f"{SOURCE_FOLDER}\\{keyword}*.xlsx")
                if not f.endswith('.crdownload')]
            if matches:
                time.sleep(0.5)
                print(f"  ⚡ File ready in {round(time.time()-start,1)}s")
                return
            time.sleep(0.5)
        print(f"  ⚠️ Timeout {timeout}s")
    else:
        time.sleep(8)

def check_and_login(driver, url) -> bool:
    print(f"  🌐 Navigating to: {url.split('/')[-1]}")
    driver.get(url); time.sleep(10)
    try:
        sign_btn = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'button[data-testid="console-okta-sign-in"]')))
        print("  🔑 Okta sign-in detected, logging in...")
        sign_btn.click(); time.sleep(2)
        try:
            WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'label[for="input36"][data-se-for-name="rememberMe"]'))).click()
            time.sleep(1)
        except TimeoutException: pass
        try:
            WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, 'input.button.button-primary[type="submit"][value="Next"]'))).click()
            time.sleep(10)
        except TimeoutException: pass
        try: driver.switch_to.alert.accept()
        except NoAlertPresentException: pass
        driver.get(url); time.sleep(5)
        print("  🎉 Login flow completed")
    except TimeoutException:
        print("  ✅ No Okta prompt — already authenticated")
    try:
        WebDriverWait(driver, LOGIN_TIMEOUT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, LOGIN_VERIFY_CSS)))
        print("  ✅ Console page confirmed loaded")
        return True
    except TimeoutException:
        raise RuntimeError(
            f"❌ Console did NOT load within {LOGIN_TIMEOUT}s\n"
            f"   Current URL: {driver.current_url}"
        )

def nice_check_timeout(driver, fallback_url, max_retry=3):
    """Nếu NICE redirect sang /timeout → reload lại URL gốc."""
    for attempt in range(max_retry):
        if "timeout" in driver.current_url.lower():
            print(f"  ⚠️ NICE session timed out — reloading (attempt {attempt+1})...")
            driver.get(fallback_url)
            time.sleep(10)
        else:
            return True
    print("  ❌ NICE session still timed out after retries")
    return False

# ── INIT DRIVER ────────────────────────────────────────────────
chrome_options = Options()
chrome_options.add_argument(r"--user-data-dir=C:/temp/new_chrome_profile")
chrome_options.add_argument(r"--profile-directory=Default")
chrome_options.add_argument("--start-maximized")
driver  = webdriver.Chrome(service=Service(CHROMEDRIVER), options=chrome_options)
wait    = WebDriverWait(driver, 15)
wait_sp = WebDriverWait(driver, 20)

print(f"\n{'═'*55}")
print(f"🚀 Bot started: {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
print(f"{'═'*55}")

try:
    # ══ STEP 1: Current Interval ═══════════════════════════════
    print("\n[1/5] Current Interval CSV")
    check_and_login(driver, URL_BREAKDOWN)
    try:
        btns = wait.until(lambda d: d.find_elements(By.CSS_SELECTOR, "button.settingsButton"))
        if not btns: raise Exception("No settingsButton found")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btns[0]); time.sleep(0.5)
        driver.execute_script("arguments[0].click();", btns[0])
        click_download_csv(driver, wait, keyword="Current Interval")
        move_files("Current Interval", DIRS["current_interval"])
    except Exception as e:
        print(f"  ❌ Step 1 failed: {e}")

    # ══ STEP 2: Logged-In Agents ═══════════════════════════════
    print("\n[2/5] Logged-In Agents CSV")
    check_and_login(driver, URL_REALTIME)
    try:
        btn = wait.until(lambda d: d.execute_script("""
            const el=Array.from(document.querySelectorAll('*')).find(e=>
                e.childNodes.length===1&&e.childNodes[0].nodeType===Node.TEXT_NODE&&
                e.textContent.trim()==='Logged-In Agents');
            if(!el)return null;
            let n=el.parentElement;
            while(n&&n!==document.body){
                const b=n.querySelectorAll('button.settingsButton');
                if(b.length===1)return b[0]; n=n.parentElement;}
            return null;"""))
        if btn is None: raise Exception("settingsButton not found")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn); time.sleep(0.5)
        driver.execute_script("arguments[0].click();", btn)
        click_download_csv(driver, wait, keyword="Logged-In Agents")
        move_files("Logged-In Agents", DIRS["current_agent"])
    except Exception as e:
        print(f"  ❌ Step 2 failed: {e}")

    # ══ STEP 3: Assigned Workitem (Connect) ════════════════════
    print("\n[3/5] Assigned Workitem (Connect) CSV")
    try:
        btn2 = wait.until(lambda d: d.execute_script("""
            const el=Array.from(document.querySelectorAll('*')).find(e=>
                e.childNodes.length===1&&e.childNodes[0].nodeType===Node.TEXT_NODE&&
                e.textContent.trim()==='Assigned Workitem (Connect)');
            if(!el)return null;
            let n=el.parentElement;
            while(n&&n!==document.body){
                const b=n.querySelectorAll('button.settingsButton');
                if(b.length===1)return b[0]; n=n.parentElement;}
            return null;"""))
        if btn2 is None: raise Exception("settingsButton not found")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", btn2); time.sleep(0.5)
        driver.execute_script("arguments[0].click();", btn2)
        click_download_csv(driver, wait, keyword="Assigned Workitem (Connect)")
        move_files("Assigned Workitem (Connect)", DIRS["lc_rawdata"])
    except Exception as e:
        print(f"  ❌ Step 3 failed: {e}")

    # ══ STEP 4: SharePoint — EN- UCP.xlsx ══════════════════════
    print("\n[4/5] SharePoint — EN- UCP.xlsx")
    driver.get(URL_SHAREPOINT); time.sleep(10)
    try:
        file_el = wait_sp.until(EC.presence_of_element_located((By.XPATH,
            "//span[contains(text(),'EN-') and contains(text(),'UCP')]"
            " | //span[contains(text(),'EN- UCP')]"
            " | //a[contains(@title,'EN-') and contains(@title,'UCP')]")))
        print(f"  ✅ Found: {file_el.text or file_el.get_attribute('title')}")
        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", file_el); time.sleep(1)
        driver.execute_script("""
            arguments[0].dispatchEvent(new MouseEvent('contextmenu',{
                bubbles:true,cancelable:true,view:window,button:2,buttons:2}));
        """, file_el); time.sleep(2)
        dl = wait_sp.until(EC.element_to_be_clickable((By.XPATH,
            "//*[text()='Download' or @aria-label='Download' or @data-automationid='download']")))
        driver.execute_script("arguments[0].click();", dl)
        print("  ✅ Clicked Download"); time.sleep(12)
        moved = False
        for fp in glob.glob(f"{SOURCE_FOLDER}\\*"):
            if fp.endswith(".crdownload"): continue
            name = os.path.basename(fp).upper()
            if "UCP" in name or ("EN" in name and ".XLSX" in name):
                if os.path.exists(DST_UCP): os.remove(DST_UCP)
                shutil.move(fp, DST_UCP)
                print(f"  📁 Moved → {os.path.basename(DST_UCP)}"); moved = True
        if not moved: print("  ⚠️ UCP file not found")
    except Exception as e:
        print(f"  ❌ Step 4 failed: {e}")

    # ══ STEP 5: NICE WFM — Agent Schedules ═════════════════════
    print(f"\n[5/5] NICE WFM — Agent Schedules ({NICE_FROM} → {NICE_TO})")
    driver.get(NICE_URL_GENERATE); time.sleep(10)

    if not nice_check_timeout(driver, NICE_URL_GENERATE):
        print("  ❌ Step 5 skipped — NICE session timeout")
    else:
        retry, found = 0, False
        while retry < 5:
            try:
                gen_link = WebDriverWait(driver, 10).until(EC.presence_of_element_located(
                    (By.XPATH, '//a[@title="Generate" and contains(@class,"sub-menu-item")]')))
                if gen_link.is_displayed():
                    print(f"  ✅ NICE loaded (attempt {retry+1})"); found = True; break
                raise Exception("not visible")
            except Exception:
                retry += 1
                print(f"  ⏳ Attempt {retry}/5 — retrying NICE...")
                time.sleep(5)
                driver.execute_script(f"window.open('{NICE_URL_GENERATE}','_blank');")
                driver.switch_to.window(driver.window_handles[-1])
                if not nice_check_timeout(driver, NICE_URL_GENERATE):
                    break

        if not found:
            print("  ❌ NICE page did not load — skipping Step 5")
        else:
            try:
                driver.get(NICE_REPORT_URL); time.sleep(5)
                if not nice_check_timeout(driver, NICE_URL_GENERATE):
                    raise RuntimeError("Session timeout on report page")

                inp_s = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.ID, "stAbsDate")))
                inp_s.clear(); inp_s.send_keys(NICE_FROM); time.sleep(1)

                inp_e = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.ID, "endAbsDate")))
                inp_e.clear(); inp_e.send_keys(NICE_TO); time.sleep(1)

                WebDriverWait(driver, 10).until(EC.element_to_be_clickable(
                    (By.XPATH, "//input[@type='submit' and @value='Generate']"))).click()
                print(f"  ✅ Generate clicked | {NICE_FROM} → {NICE_TO}")
                time.sleep(3)

                try: driver.switch_to.alert.accept()
                except NoAlertPresentException: pass

                print("  ⏳ Waiting 60s for generation...")
                time.sleep(60)

                driver.get(NICE_URL_VIEW)
                if not nice_check_timeout(driver, NICE_URL_GENERATE):
                    raise RuntimeError("Session timeout on view page")

                wait_n = WebDriverWait(driver, 20)
                iframe = wait_n.until(EC.presence_of_element_located(
                    (By.CLASS_NAME, "legacy-wrapper")))
                driver.switch_to.frame(iframe)
                wait_n.until(EC.element_to_be_clickable(
                    (By.XPATH, '//input[@type="submit" and @value="Refresh"]'))).click()
                time.sleep(10)
                wait_n.until(EC.element_to_be_clickable(
                    (By.LINK_TEXT, "Agent Schedules"))).click()
                print("  ✅ Clicked 'Agent Schedules'")
                driver.switch_to.default_content()
                time.sleep(15)

                dst_path = os.path.join(DIRS["current_iex"], NICE_FILE)
                patterns = (
                    glob.glob(f"{SOURCE_FOLDER}\\Agent Schedules*.csv")  +
                    glob.glob(f"{SOURCE_FOLDER}\\Agent Schedules*.xlsx") +
                    glob.glob(f"{SOURCE_FOLDER}\\agent*schedule*.xlsx")  +
                    glob.glob(f"{SOURCE_FOLDER}\\report*.xlsx")          +
                    glob.glob(f"{SOURCE_FOLDER}\\report*.csv")
                )
                moved = False
                for fp in sorted(patterns, key=os.path.getmtime, reverse=True):
                    if fp.endswith(".crdownload"): continue
                    if os.path.exists(dst_path): os.remove(dst_path)
                    shutil.move(fp, dst_path)
                    print(f"  📁 {os.path.basename(fp)} → {NICE_FILE}"); moved = True; break
                if not moved:
                    print(f"  ⚠️ No report file found in {SOURCE_FOLDER}")

            except Exception as e:
                print(f"  ❌ Step 5 failed: {e}")

except RuntimeError as e:
    print(f"\n🚨 FATAL: {e}")
except WebDriverException as e:
    print(f"\n🚨 WEBDRIVER ERROR: {e}")
finally:
    driver.quit()
    print(f"\n{'═'*55}")
    print(f"✅ Bot finished: {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
    print(f"{'═'*55}")


═══════════════════════════════════════════════════════
🚀 Bot started: 01-Aug-2026 15:06:16
═══════════════════════════════════════════════════════

[1/5] Current Interval CSV
  🌐 Navigating to: agentBreakdownRealtimeDashboard
  ✅ No Okta prompt — already authenticated
  ✅ Console page confirmed loaded
  ✅ Clicked Download CSV
  ⚡ File ready in 1.0s
  📁 Moved: Current Interval-Sat Aug 01 2026 15_06_38 GMT+0700 (Indochina Time).csv

[2/5] Logged-In Agents CSV
  🌐 Navigating to: agentRealtime
  ✅ No Okta prompt — already authenticated
  ✅ Console page confirmed loaded
  ✅ Clicked Download CSV
  ⚡ File ready in 1.0s
  📁 Moved: Logged-In Agents-Sat Aug 01 2026 15_07_01 GMT+0700 (Indochina Time).csv

[3/5] Assigned Workitem (Connect) CSV
  ✅ Clicked Download CSV
  ⚡ File ready in 1.0s
  📁 Moved: Assigned Workitem (Connect)-Sat Aug 01 2026 15_07_03 GMT+0700 (Indochina Time).csv

[4/5] SharePoint — EN- UCP.xlsx
  ✅ Found: EN- UCP.xlsx
  ✅ Clicked Download
  📁 Moved → EN- UCP.xlsx

[5/5] NICE

In [31]:
import openpyxl
import polars as pl
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

UCP_FILE = r"C:\Users\huuchinh.nguyen\Concentrix Corporation\WFM-Expedia-HCM - Branding files\Rawdata\CAPTURE\EN- UCP.xlsx"
TZ_VNT   = ZoneInfo("Asia/Ho_Chi_Minh")
TZ_PST   = ZoneInfo("America/Los_Angeles")

def read_range(wb, sheet_name, header_row=2, data_start=3, data_end=50):
    ws      = wb[sheet_name]
    headers = [str(ws.cell(row=header_row, column=c).value or f"Col_{c}").strip()
               for c in range(7, 11)]
    rows = []
    for row in ws.iter_rows(min_row=data_start, max_row=data_end, min_col=7, max_col=10):
        rows.append([cell.value for cell in row])
    df = pl.DataFrame(rows, schema=headers, orient="row")
    return df.filter(pl.any_horizontal(pl.all().is_not_null()))

def gen_intervals(n_rows):
    today    = datetime.now(TZ_PST).date()
    base_pst = datetime(today.year, today.month, today.day, 0, 0, tzinfo=TZ_PST)
    vnt_list, pst_list = [], []
    for i in range(n_rows):
        pst = base_pst + timedelta(minutes=30*i)
        vnt = pst.astimezone(TZ_VNT)
        pst_list.append(pst.strftime("%H:%M"))
        vnt_list.append(vnt.strftime("%H:%M"))
    return vnt_list, pst_list

def attach_intervals(df, lob):
    vnt_list, pst_list = gen_intervals(len(df))
    return df.with_columns([
        pl.Series("VNT", vnt_list),
        pl.Series("PST", pst_list),
        pl.lit(lob).alias("LOB"),
    ]).select(["LOB","VNT","PST"] + df.columns)

wb = openpyxl.load_workbook(UCP_FILE, data_only=True)
print(f"Sheets: {wb.sheetnames}")

df_nl = attach_intervals(read_range(wb, "NL Chat"), "NL Chat")
df_lg = attach_intervals(read_range(wb, "LG Chat"), "LG Chat")

df_ucp = pl.concat([df_lg, df_nl], how="diagonal_relaxed").sort(["LOB","PST"])
print(f"df_ucp: {df_ucp.shape}")
print(df_ucp)

Sheets: ['1st Aug IC Action Plan', 'Cairo PLS Names', 'Voice LG Names', 'Voice NLG Names', 'Sheet3', 'Sheet1', 'Sheet2', 'Chat LIO Names', 'Chat NLG Names', 'Chat LG Names', 'Sheet4', 'Voice NL (2)', 'Chat LIO', 'Sheet5', 'Real time overage and leakage', 'RCA', 'French', 'Spanish', ' Chat LIO', 'Variance All LOB', 'Cross Skilling Metrix SA%', 'NL Voice', 'NL Chat', 'LG Voice', 'LG Chat', ' Chat LIO ', 'EN PLS NLV', 'Halifax(RBC)-PLS', 'Movement', 'NL Chat (2)', 'Hp French', 'HP IT', 'HP GR', 'HP TR', 'HP DE', 'EN PLS Chat', 'EN PLS LG', 'China-PLS', 'Interval view']
df_ucp: (96, 7)
shape: (96, 7)
┌─────────┬───────┬───────┬───────┬─────────┬─────────┬──────┐
│ LOB     ┆ VNT   ┆ PST   ┆ Cairo ┆ Vietnam ┆ Kolkata ┆ Pune │
│ ---     ┆ ---   ┆ ---   ┆ ---   ┆ ---     ┆ ---     ┆ ---  │
│ str     ┆ str   ┆ str   ┆ f64   ┆ f64     ┆ f64     ┆ i64  │
╞═════════╪═══════╪═══════╪═══════╪═════════╪═════════╪══════╡
│ LG Chat ┆ 14:00 ┆ 00:00 ┆ 0.0   ┆ 26.34   ┆ 8.67    ┆ 0    │
│ LG Chat ┆ 14:30 